# Demo 4 – Dynamische Inhalte mit Selenium
[Start](../README.md) · [Demos](README.md) · [Task 2](../tasks/02_dynamischer_katalog/README.md)

Eine Beschaffungsliste soll alle Produkte eines Katalogs enthalten. Im Browser sind Produkte sichtbar, aber ein erster HTML-Abruf enthält keine Tabellenzeilen. Wir prüfen **denselben HTML-Inhalt** mit und ohne JavaScript-Ausführung.

**Lernziele:** Du unterscheidest HTTP-Antwort und gerendertes DOM, wartest auf einen fachlich sinnvollen Zustand, betätigst einen Button und kombinierst Selenium mit BeautifulSoup. Du kontrollierst die Vollständigkeit vor dem Export.

**Voraussetzung:** `00_setup_check.ipynb` meldet `SETUP OK`; lokal läuft `docker compose up -d`. In Codespaces ist der Browser-Dienst bereits eingerichtet. Alle sechs Produkte sind synthetisch, es gibt keinen externen Live-Abruf. Führe die Zellen von oben nach unten aus.

## 1. Werkzeuge und Projektpfad
`requests` ist der HTTP-Client. BeautifulSoup parst HTML. Selenium steuert einen echten Browser, der JavaScript ausführt. Der kleine Helfer `browser_session` übernimmt nur die Verbindung zum vorhandenen Container und das zuverlässige Beenden der Sitzung; die eigentlichen Lernschritte bleiben hier sichtbar.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/webscraping_workshop/checks.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Öffne das Notebook innerhalb des Workshop-Repos.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import json
from datetime import datetime, timezone
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webscraping_workshop.checks import fixture_server
from webscraping_workshop.runtime import browser_session, html_data_url

## 2. Zuerst die HTTP-Antwort untersuchen
**Vorhersage:** Wie viele Tabellenzeilen wird BeautifulSoup finden? Öffne bei Bedarf auch `data/teaching/dynamic_products.html` direkt im lokalen Browser oder lies die Datei im Editor. Vergleiche den Seitenquelltext mit dem DOM in den Entwicklertools.

In [2]:
# Nur Python ruft diesen kurzlebigen HTTP-Server auf.
with fixture_server(ROOT / "data/teaching") as base_url:
    with requests.Session() as session:
        session.trust_env = False  # Loopback nicht über einen Firmenproxy schicken.
        response = session.get(f"{base_url}/dynamic_products.html", timeout=(3, 10))
        response.raise_for_status()
        response.encoding = "utf-8"
        raw_html = response.text

source_soup = BeautifulSoup(raw_html, "lxml")
raw_rows = source_soup.select("#products tbody tr")
print("Tabellenzeilen in der HTTP-Antwort:", len(raw_rows))
assert len(raw_rows) == 0, "Die Lehrseite sollte anfangs einen leeren Tabellenkörper haben."

Tabellenzeilen in der HTTP-Antwort: 0


Die leere Liste ist hier ein **beabsichtigter Befund**, kein Installationsfehler. Der Tabellenkörper ist vorhanden, aber leer. JavaScript ergänzt später drei Produkte und nach einem Klick drei weitere.

Das Beispiel enthält seine synthetischen Daten direkt im Skript, damit es ohne externe Dienste funktioniert. Diese eingebetteten strukturierten Daten wären ebenfalls eine mögliche Quelle. Bei realen Seiten prüfst Du im Netzwerk-Tab, ob eine zulässige API oder ein strukturierter Download verfügbar ist. **Keine Zeilen allein beweisen noch nicht, dass Selenium nötig ist:** Auch falsche Selektoren, eine geänderte Seite oder eine Fehlerseite kommen als Ursache infrage.

## 3. Genau dieses HTML im Browser ausführen
Wir übertragen `raw_html` als `data:`-URL. So erhält der Browser im Docker-Container exakt denselben Inhalt wie BeautifulSoup. Er muss nicht auf den lokalen Python-HTTP-Server zugreifen. Eine reale Website würdest Du mit `browser.get("https://...")` öffnen.

`WebDriverWait` wartet auf einen Zustand statt auf eine geschätzte Pause. Erst wenn die ersten drei Zeilen vorhanden sind, wird der Button betätigt. Danach warten wir auf die Gesamtzahl aus `data-total`.

In [3]:
output_dir = ROOT / "data/output/demo04_selenium_sample"
output_dir.mkdir(parents=True, exist_ok=True)

with browser_session(ROOT) as browser:
    browser.get(html_data_url(raw_html))
    wait = WebDriverWait(browser, 15)
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "#products tbody tr")) == 3)
    first_batch_html = browser.page_source
    print("Nach dem ersten Rendern:", len(browser.find_elements(By.CSS_SELECTOR, "#products tbody tr")))

    expected_total = int(browser.find_element(By.ID, "products").get_attribute("data-total"))
    wait.until(EC.element_to_be_clickable((By.ID, "load-more"))).click()
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "#products tbody tr")) == expected_total)
    wait.until(lambda d: not d.find_element(By.ID, "load-more").is_enabled())
    rendered_html = browser.page_source
    # Selenium kann auch direkt Daten aus DOM-Elementen lesen.
    selenium_names = [e.text for e in browser.find_elements(By.CSS_SELECTOR, "#products tbody .name")]
    print(browser.find_element(By.ID, "status").text)
    if not browser.save_screenshot(str(output_dir / "browser.png")):
        raise RuntimeError("Screenshot konnte nicht gespeichert werden.")
# Auch bei einem Fehler wird die Browser-Sitzung durch den Kontextmanager beendet.

Nach dem ersten Rendern: 3
6 von 6 Produkten geladen


## 4. Gerendertes HTML mit BeautifulSoup auswerten
BeautifulSoup ist weiterhin nützlich: Sein Eingabetext enthält jetzt die vom Browser ergänzten Elemente. Selenium wird für JavaScript und Interaktionen eingesetzt, BeautifulSoup für die Extraktion.

In [4]:
def extract_basic_products(html):
    soup = BeautifulSoup(html, "lxml")
    table = soup.select_one("table#products[data-mode='synthetic']")
    if table is None:
        raise ValueError("Erwartete Produkttabelle fehlt; Quelle oder Selektor prüfen.")
    records = []
    for row in table.select("tbody tr"):
        name = row.select_one(".name")
        price = row.select_one(".price")
        if name is None or price is None or not row.get("data-id"):
            raise ValueError("Ein Produkt enthält nicht alle erwarteten Felder.")
        records.append({"id": row["data-id"], "name": name.get_text(strip=True),
                        "price_chf": float(price.get_text(strip=True))})
    if not records:
        raise ValueError("Keine Produkte: Ladezustand, Selektor und Quelle prüfen.")
    return pd.DataFrame(records).sort_values("id").reset_index(drop=True)

In [5]:
first_batch = extract_basic_products(first_batch_html)
products = extract_basic_products(rendered_html)
assert len(first_batch) == 3
assert len(products) == expected_total == 6
assert products["id"].is_unique
assert products["name"].str.len().gt(0).all()
assert products["price_chf"].gt(0).all()
assert set(products["name"]) == set(selenium_names)
products

,id,name,price_chf
0,P101,Adapter,29.9
1,P102,Sensor,39.0
2,P103,Kabel,9.5
3,P104,Hub,49.0
4,P105,Schalter,24.5
5,P106,Lampe,19.0


## 5. Ergebnis und Herkunft speichern
Ein Browser-Screenshot hilft bei der Fehlersuche, ersetzt aber keine Datenprüfung. Wir speichern zusätzlich eine CSV und Herkunftsmetadaten. Die Dateien liegen im ignorierten Ausgabeordner; ein erneuter Lauf ersetzt die Ergebnisse dieses Beispiels.

In [6]:
products.to_csv(output_dir / "products.csv", index=False, encoding="utf-8")
metadata = {
    "mode": "sample", "synthetic": True,
    "source": "data/teaching/dynamic_products.html",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "method": "requests -> Selenium render + click -> BeautifulSoup",
    "raw_html_rows": len(raw_rows), "first_batch_rows": len(first_batch), "exported_rows": len(products),
    "currency": "CHF",
}
(output_dir / "provenance.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
reloaded = pd.read_csv(output_dir / "products.csv")
pd.testing.assert_frame_equal(products, reloaded)
print("DEMO 4 OK: 0 Zeilen im Ausgangs-HTML, 3 nach JavaScript, 6 nach dem Klick.")

DEMO 4 OK: 0 Zeilen im Ausgangs-HTML, 3 nach JavaScript, 6 nach dem Klick.


## Einordnen und übertragen

| Beobachtung | Nächster sinnvoller Schritt |
|---|---|
| API oder strukturierter Export verfügbar | Zugriff und Bedingungen prüfen; strukturiert abrufen |
| Benötigte Daten bereits im HTTP-HTML | Requests und BeautifulSoup verwenden |
| Daten erscheinen erst durch JavaScript/Interaktion | Netzwerkantworten prüfen; bei Bedarf Selenium einsetzen |
| Browser zeigt eine Sperre oder unerwartete Fehlerseite | Ursache klären; Zugriffsbeschränkungen nicht umgehen |

**Besprich:**
1. Weshalb wäre ein Export unmittelbar nach dem ersten Rendern fachlich falsch?
2. Weshalb löst `time.sleep(2)` das Problem weniger zuverlässig als eine Zustandsprüfung?
3. Weshalb kann BeautifulSoup auch nach dem Einsatz von Selenium sinnvoll sein?
4. Welche Information bräuchtest Du, um einen API-Abruf einem Browserlauf vorzuziehen?
5. Was wäre bei einer realen, während des Abrufs veränderlichen Gesamtzahl zusätzlich zu beachten?

**Quellen und Bezug:** Aufbauend auf dem BeautifulSoup-/Selenium-Vergleich der Vorjahresmaterialien. Die Yahoo-Seite wird hier durch ein kontrolliertes Lehrbeispiel ersetzt; die Methodenwahl wird nicht pauschal aus dem Namen einer Website abgeleitet. [Selenium: Waiting Strategies](https://www.selenium.dev/documentation/webdriver/waits/) · [Remote WebDriver](https://www.selenium.dev/documentation/webdriver/drivers/remote_webdriver/) · [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/). Dokumentation geprüft: 30.09.2026.

[↑ Demos](README.md) · [Weiter zu Task 2](../tasks/02_dynamischer_katalog/README.md)